In [10]:
# initilization(simple random)
def initialize_parameter(layer_dimension):
    parameters = {}
    L = len(layer_dimension)
    for l in range(1, L):
        parameters["W" + str(l)] = np.random.randn(layer_dimension[l], layer_dimension[l-1])
        parameters["b"+ str(l)] = np.zeros((layer_dimension[l], 1))
    
        assert(parameters["W" + str(l)].shape == (layer_dimension[l], layer_dimension[l-1]))
        assert(parameters["b" + str(l)].shape == (layer_dimension[l], 1))
    return parameters

In [11]:
parameters = initialize_parameter([2, 8, 8, 1])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
parameters["W1"].shape 

W1 = [[ 1.78862847  0.43650985]
 [ 0.09649747 -1.8634927 ]
 [-0.2773882  -0.35475898]
 [-0.08274148 -0.62700068]
 [-0.04381817 -0.47721803]
 [-1.31386475  0.88462238]
 [ 0.88131804  1.70957306]
 [ 0.05003364 -0.40467741]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.54535995 -1.54647732  0.98236743 -1.10106763 -1.18504653 -0.2056499
   1.48614836  0.23671627]
 [-1.02378514 -0.7129932   0.62524497 -0.16051336 -0.76883635 -0.23003072
   0.74505627  1.97611078]
 [-1.24412333 -0.62641691 -0.80376609 -2.41908317 -0.92379202 -1.02387576
   1.12397796 -0.13191423]
 [-1.62328545  0.64667545 -0.35627076 -1.74314104 -0.59664964 -0.58859438
  -0.8738823   0.02971382]
 [-2.24825777 -0.26776186  1.01318344  0.85279784  1.1081875   1.11939066
   1.48754313 -1.11830068]
 [ 0.84583341 -1.86088953 -0.6028851  -1.91447204  1.04814751  1.33373782
  -0.19741468  1.77464503]
 [-0.67472751  0.15061687  0.1529457  -1.06419527  0.43794661  1.93897846
  -1.02493087  0.89933845]
 [-0.15450685

(8, 2)

In [12]:
def activation(A_prev, W, b, activation):
    if activation == "sigmoid":
        Z = np.dot(W, A_prev) + b 
        linear_cache = (A_prev, W, b)
        A  = 1 / (1 + np.exp(-Z))
        activation_cache = (A)
    
    elif activation == "relu":
        Z = np.dot(W, A_prev) + b
        linear_cache = (A_prev, W, b)
        A = np.maximum(0, Z)
        activation_cache = (A)
    
    cache = (linear_cache,activation_cache)
    
    return A, cache

In [13]:
def forward_prop(X, parameters):
    caches = []
    A = X
    L = len(parameters) // 2
    
    for l in range (1,L):
        A_prev = A 
        A, cache = activation(A_prev, parameters["W" + str(l)], parameters["b" + str(l)], "relu")
        caches.append(cache)
        
    last_layer, cache_L = activation (A, parameters["W" + str(L)], parameters["b" + str(L)], "sigmoid")
    caches.append(cache_L)

    return last_layer, caches 

In [14]:
X = Generate_dots(1.5).T
labels = labeling_circle_boundry(1, X.T)

predicts, test_cache = forward_prop(X, parameters)
predicts.shape 
#بعدا ساختار cache رو یه فری به حالش بکم و بهینه ترش بکن 
#تکذاذ مسخره اکتیویشن البته کار میکند 

(1, 2000)

In [15]:
labels.shape 

(2000,)

In [16]:
def cost_function(labels, predicts):
    m = labels.shape[0]
    cost = np.sum((labels.T * np.log(predicts)) + ((1-labels.T)*np.log(1- predicts)))
    cost /= -m
    cost = np.squeeze(cost)
    return cost

In [17]:
len(test_cache)

3

In [18]:
test_cost  = cost_function(labels, predicts)
test_cost

0.9567819222196691

In [ ]:
def back_prop(last_layer, labels, cache):
    gradients = {}
    L = len(cache)
    m = last_layer.shape[1]
    last_layer = last_layer.reshape(1, -1)
    labels = labels.reshape(1, -1)
    dAL = -np.divide(labels, last_layer) + np.divide((1-labels), (1-last_layer))
    
    #sigmoid layer 
    dZ = dAL * last_layer * (1 - last_layer)
    dA_prevL = np.dot(cache[L-1][0][1].T,  dZ) #cache ((A-prev, w, b), nonlinearity )
    dW = 1/m * np.dot(dZ, cache[L-1][0][0].T)   #A[l-1]
    db =1/m * (np.sum(dZ, axis=1, keepdims=True))
    gradients["dA" + str(L-1)] = dA_prevL
    gradients["dW" + str(L)] = dW
    gradients["db" + str(L)] = db
   
   # relu layers 
    for l in reversed(range(L-1)) : 
        current_cache = cache[l]
        dZ = dA_prevL * (current_cache[1] > 0)
        dA_prevL = np.dot(current_cache[0][1].T, dZ)
        dW = 1/m * np.dot(dZ, current_cache[0][0].T)
        db = 1/m * (np.sum(dZ))
        
        gradients["dA" + str(L-1)] = dA_prevL
        gradients["dW" + str(L)] = dW
        gradients["db" + str(L)] = db
    
    return gradients
   

In [20]:
back_prop(predicts, labels, test_cache)

{'dA2': array([[ 9.60110524e-01, -5.27753471e-01, -2.72843631e+00, ...,
         -2.27613583e-04,  5.00775852e-01, -3.90076908e+00],
        [ 3.55517340e+00,  1.48911716e+00, -9.78908511e-01, ...,
          1.03637026e-02,  1.38629032e-01, -1.39951812e+00]]),
 'dW3': array([[ 0.06728987,  0.00884925],
        [-0.03887023, -0.0799173 ],
        [ 0.08747929, -0.05475433],
        [ 0.01346851, -0.0200827 ],
        [ 0.00033852, -0.03040535],
        [-0.2412725 ,  0.19073266],
        [ 0.07199231, -0.16938687],
        [ 0.04210834, -0.08498124]]),
 'db3': 0.42451042775521486}